# Results export for the paper (no training, about 10 to 20 minutes)

This notebook does two things and nothing else:
1. **Copies the saved predictions** of all models (internal test, BraTS, tumor-free, BDNeuro, Mendeley, BRISC),
   the cross validation records and the lists of evaluated images. With these, all statistical tests are
   recomputed against the proposed model **ROI-0.6+TT** (`ROI-m06-Transplant`).
2. **Computes real heat maps** (two-level LayerCAM, the method of Section 2.6.3) for 4 models on the same
   24 test slices used in Figs. 2 and 3, averaged over the 5 fold models.

**How to run**
1. Accelerator: **GPU T4** (None also works, it is only slower).
2. Add Input (same as the figure assets notebook): the Figshare PNG dataset, your **Run 6 results** dataset,
   and the datasets that hold the **model checkpoints** (`.pt`). Also add the **BraTS 2D slices** and
   **BraTS tumor-free slices** datasets (they hold `manifest.csv` and `manifest_free.csv`).
3. Click **Run All**.
4. Download `results_export.zip` from the Output panel (right side, /kaggle/working) and send it to Claude.

In [ ]:
import os, json, shutil, zipfile, time
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn, torch.nn.functional as F, timm
from PIL import Image

INPUT = Path(os.environ.get("EXPORT_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("EXPORT_WORK", "/kaggle/working")); OUT = WORK / "results_export"
shutil.rmtree(OUT, ignore_errors=True); OUT.mkdir(parents=True)
IMG, K = 224, 3
CLASSES = ["glioma", "meningioma", "pituitary"]
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
SEG_NAME, MARGIN, MIN_PIX, N_PER_CLASS = "MTL-Seg", 0.6, 20, 8
CAM_MODELS = ["CNX-Plain", "CAM-Guide", "CNX-Transplant", "ROI-m06-Transplant"]   # Baseline, CAM-Guide, TT, ROI-0.6+TT
ROI_MODELS = {"ROI-m06-Transplant"}
t_start = time.time()

def find_all(pattern):
    return sorted(INPUT.rglob(pattern))

def find_one(name, required=True):
    hits = sorted(INPUT.rglob(name), key=lambda p: p.stat().st_size, reverse=True)
    if not hits and required:
        raise FileNotFoundError(f"'{name}' not found under {INPUT}. Add the dataset that contains it (see the list at the top).")
    return hits[0] if hits else None

In [ ]:
# ---- 1. Copy predictions, run records, out-of-fold logits and the lists of evaluated images ----
def copy_group(pattern, sub, parent_name=None):
    (OUT / sub).mkdir(exist_ok=True); seen = set()
    for f in find_all(pattern):
        if parent_name and f.parent.name != parent_name:
            continue
        if f.name in seen:
            continue
        seen.add(f.name); shutil.copy(f, OUT / sub / f.name)
    return len(seen)

n_pred = copy_group("*_s*_f*_*.npy", "preds", "preds")
n_runs = copy_group("*_s*_f*.json", "runs", "runs")
n_oof = copy_group("*_s*_f*_logits.npy", "oof", "oof")
n_rel = copy_group("*_s*_f*.npz", "reliance", "reliance")
print(f"predictions: {n_pred} files | run records: {n_runs} | out-of-fold logits: {n_oof} | reliance caches: {n_rel}")
if n_pred == 0 or n_runs == 0:
    print("WARNING: predictions or run records not found. Add the Run 6 RESULTS dataset (the one with the outputs folder).")

lists = {"split_3class.csv": True, "overlap_audit.csv": True, "manifest_free.csv": False}
(OUT / "lists").mkdir(exist_ok=True)
for name, req in lists.items():
    f = find_one(name, required=False)
    print(f"{name:20s}: {f if f else 'NOT FOUND' + (' (required)' if req else '')}")
    if f: shutil.copy(f, OUT / "lists" / name)
bm = [h for h in find_all("manifest.csv") if (h.parent / "gli").exists() or (h.parent / "men").exists()]
print(f"{'BraTS manifest.csv':20s}: {bm[0] if bm else 'NOT FOUND (add the BraTS 2D slices dataset)'}")
if bm: shutil.copy(bm[0], OUT / "lists" / "manifest_brats.csv")
print("models with test predictions:", sorted({f.name.split("_s")[0] for f in (OUT / "preds").glob("*_test.npy")}))

In [ ]:
# ---- 2. Model code (identical to Run 6) ----
SPLIT, MASKF = find_one("split_3class.csv"), find_one("figshare_masks_224.npz")
_IDX = {}
for p in INPUT.rglob("*.png"):
    _IDX.setdefault(p.name, str(p))
def resolve(path):
    return path if os.path.exists(path) else _IDX.get(Path(path).name)
df = pd.read_csv(SPLIT); df["path"] = df.filepath.map(resolve); df = df.dropna(subset=["path"]).reset_index(drop=True)
z = np.load(MASKF, allow_pickle=True)
MASKS = {str(s): np.unpackbits(b)[:IMG * IMG].reshape(IMG, IMG).astype(bool) for s, b in zip(z["stems"], z["bits"])}
df = df[df.path.map(lambda p: Path(p).stem in MASKS)].reset_index(drop=True)
def load_u8(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("L").resize((IMG, IMG), Image.BICUBIC), np.uint8)
def mask_of(path): return MASKS[Path(path).stem]

class GuidedConvNeXt(nn.Module):
    def __init__(self, num_classes=K, seg=False):
        super().__init__()
        self.use_seg = seg
        self.backbone = timm.create_model("convnext_tiny", pretrained=False, features_only=True, out_indices=(1, 2, 3))
        c1, c2, c3 = self.backbone.feature_info.channels()
        self.cam_id = nn.Identity(); self.cam_id2 = nn.Identity()
        if seg:
            self.lat3, self.lat2, self.lat1 = nn.Conv2d(c3, 128, 1), nn.Conv2d(c2, 128, 1), nn.Conv2d(c1, 128, 1)
            self.seg_head = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(), nn.Conv2d(64, 1, 1))
        self.head_norm = nn.LayerNorm(c3)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(c3, num_classes))
    def forward(self, x, return_seg=False):
        f1, f2, f3 = self.backbone(x); seg = None
        if self.use_seg:
            d = self.lat3(f3)
            d = self.lat2(f2) + F.interpolate(d, size=f2.shape[-2:], mode="bilinear", align_corners=False)
            d = self.lat1(f1) + F.interpolate(d, size=f1.shape[-2:], mode="bilinear", align_corners=False)
            seg = self.seg_head(d)
        logits = self.head(self.head_norm(f3.mean((2, 3))))
        return (logits, seg) if return_seg else logits

def box_from_mask(mask, min_pixels=MIN_PIX):
    m = np.asarray(mask, np.uint8)
    if m.sum() < min_pixels:
        return None
    n, lab, st, _ = cv2.connectedComponentsWithStats(m)
    k = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    if st[k, cv2.CC_STAT_AREA] < min_pixels:
        return None
    x, y, w, h = (int(v) for v in st[k, :4])
    return x, y, w, h, lab == k

def window(box, margin=MARGIN):
    x, y, w, h, _ = box; s = max(max(w, h) * (1 + 2 * margin), 32); cx, cy = x + w / 2, y + h / 2
    return int(cx - s / 2), int(cy - s / 2), int(cx + s / 2), int(cy + s / 2)
def crop_roi(img, box, margin=MARGIN):
    return np.asarray(Image.fromarray(img).crop(window(box, margin)).resize((IMG, IMG), Image.BICUBIC), np.uint8)


DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def to_batch(arrs):
    x = torch.from_numpy(np.stack(arrs)).to(DEV).float().div_(255).unsqueeze(1).repeat(1, 3, 1, 1)
    return (x - torch.tensor(MEAN, device=DEV).view(1, 3, 1, 1)) / torch.tensor(STD, device=DEV).view(1, 3, 1, 1)
def load(name, fold, seg=False):
    ck = find_one(f"{name}_s0_f{fold}.pt", required=False)
    if ck is None:
        return None
    m = GuidedConvNeXt(seg=seg).to(DEV)
    sd = torch.load(ck, map_location=DEV); m.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()}, strict=True)
    return m.eval()

class LayerCAM2:
    """Two-level LayerCAM (Jiang et al., 2021) on the 28x28 and 14x14 stages, as in Run 6 (Eq. 9)."""
    def __init__(self, model):
        self.model, self.feats = model, None
        self.h = model.backbone.register_forward_hook(lambda mod, inp, out: setattr(self, "feats", out))
    def __call__(self, x, target):
        self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            out = self.model(x.float()); f1, f2 = self.feats[0], self.feats[1]
            g1, g2 = torch.autograd.grad(out[0, int(target)], [f1, f2])
        maps = []
        for A, G in [(f1, g1), (f2, g2)]:
            c = F.relu((F.relu(G) * A).sum(1, keepdim=True)).detach().float()
            c = F.interpolate(c, (IMG, IMG), mode="bilinear", align_corners=False)[0, 0]
            maps.append((c - c.min()) / (c.max() - c.min() + 1e-8))
        cam = (maps[0] + maps[1]) / 2; cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.cpu().numpy(), torch.softmax(out.detach().float(), 1)[0].cpu().numpy()
    def close(self):
        self.h.remove()

def paste_back(cam_crop, win):
    x0, y0, x1, y1 = win; big = np.zeros((IMG, IMG), np.float32)
    cs = np.asarray(Image.fromarray(cam_crop.astype(np.float32)).resize((x1 - x0, y1 - y0), Image.BILINEAR))
    xa, ya, xb, yb = max(x0, 0), max(y0, 0), min(x1, IMG), min(y1, IMG)
    if xb > xa and yb > ya:
        big[ya:yb, xa:xb] = cs[ya - y0:yb - y0, xa - x0:xb - x0]
    return big

In [ ]:
# ---- 3. Heat maps on the same 24 test slices as Figs. 2 and 3 (same selection, seed 7) ----
rng = np.random.default_rng(7); cand = []
test = df[df.split == "test"]
for c in CLASSES:
    sub = test[test.label4 == c]
    ok = [p for p in sub.path if 0.006 <= mask_of(p).mean() <= 0.07]
    ok = ok if len(ok) >= N_PER_CLASS else list(sub.path)
    cand += [(c, p) for p in rng.permutation(ok)[:N_PER_CLASS]]
arrs = [load_u8(p) for _, p in cand]; targets = [CLASSES.index(c) for c, _ in cand]
print("slices:", len(arrs), "| first stems:", [Path(p).stem for _, p in cand[:3]])

segmaps = {}
with torch.no_grad():
    for f in range(1, 6):
        seg = load(SEG_NAME, f, seg=True)
        if seg is None:
            print(f"MTL-Seg fold {f} checkpoint missing"); continue
        _, sl = seg(to_batch(arrs), return_seg=True)
        segmaps[f] = torch.sigmoid(F.interpolate(sl.float(), (IMG, IMG), mode="bilinear", align_corners=False))[:, 0].cpu().numpy()
        del seg

res = {"stems": np.array([Path(p).stem for _, p in cand]), "classes": np.array([c for c, _ in cand]), "targets": np.array(targets)}
for name in CAM_MODELS:
    cams = np.zeros((len(arrs), IMG, IMG), np.float32); probs = np.zeros((len(arrs), K), np.float32)
    nf = np.zeros(len(arrs)); folds_used = 0; t0 = time.time()
    for f in range(1, 6):
        m = load(name, f)
        if m is None:
            print(f"  {name} fold {f}: checkpoint missing"); continue
        folds_used += 1; eng = LayerCAM2(m)
        for i, a in enumerate(arrs):
            if name in ROI_MODELS:
                if f not in segmaps: continue
                b = box_from_mask(segmaps[f][i] >= 0.5)
                if b is None: continue
                cam, pr = eng(to_batch([crop_roi(a, b)]), targets[i]); cam = paste_back(cam, window(b))
            else:
                cam, pr = eng(to_batch([a]), targets[i])
            cams[i] += cam; probs[i] += pr; nf[i] += 1
        eng.close(); del m
        if DEV.type == "cuda": torch.cuda.empty_cache()
    ok = nf > 0
    cams[ok] /= nf[ok][:, None, None]; probs[ok] /= nf[ok][:, None]
    res[f"{name}__cam"] = cams.astype(np.float16); res[f"{name}__probs"] = probs; res[f"{name}__folds"] = nf
    print(f"{name:20s}: {folds_used} fold models | {time.time() - t0:5.0f} s | tumor found on {int(ok.sum())}/{len(arrs)} slices")
np.savez_compressed(OUT / "heatmaps_24.npz", **res)

In [ ]:
# ---- 4. Zip ----
zp = WORK / "results_export.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT.rglob("*")):
        if f.is_file(): zf.write(f, f.relative_to(WORK))
print(f"\nDONE in {(time.time() - t_start) / 60:.1f} min. results_export.zip = {zp.stat().st_size / 1e6:.1f} MB")
print("Download it from the Output panel (right side, /kaggle/working) and send it to Claude.")